In [1]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

In [2]:
file_path = "brown_nouns.txt"

with open(file_path, "r") as file:
    words = [line.strip() for line in file if line.strip()]

print("Total entries:", len(words))
print("First 10 words:", words[:10])

Total entries: 202793
First 10 words: ['investigation', 'primary', 'election', 'evidence', 'irregularities', 'place', 'jury', 'presentments', 'charge', 'election']


In [3]:
noun_lexicon = set(words)

print("Total entries:", len(words))
print("Unique nouns:", len(noun_lexicon))

Total entries: 202793
Unique nouns: 19287


In [4]:
dfa_transition = {
    "q0": {
        "lowercase": "q1",
        "other": "q_dead"
    },
    
    "q1": {
        "lowercase": "q1",
        "other": "q_dead"
    },
    
    "q_dead": {
        "lowercase": "q_dead",
        "other": "q_dead"
    }
}

accepting_state = "q1"

print("DFA Transition Table")
print("--------------------")
print("State     lowercase     other")
print("q0        q1            q_dead")
print("q1        q1            q_dead")
print("q_dead    q_dead        q_dead")

DFA Transition Table
--------------------
State     lowercase     other
q0        q1            q_dead
q1        q1            q_dead
q_dead    q_dead        q_dead


In [5]:
def dfa_check(word):
    state = "q0"

    for i, ch in enumerate(word):
        if "a" <= ch <= "z":
            symbol = "lowercase"
        else:
            symbol = "other"

        state = dfa_transition[state][symbol]

    if state == accepting_state:
        return "Accepted"
    else:
        return "Not Accepted"

In [6]:
test_words = [
    "cat",
    "dog",
    "a",
    "zebra",
    "dog1",
    "1dog",
    "DogHouse",
    "Dog_house",
    " cats"
]

for word in test_words:
    print(word, "=", dfa_check(word))

cat = Accepted
dog = Accepted
a = Accepted
zebra = Accepted
dog1 = Not Accepted
1dog = Not Accepted
DogHouse = Not Accepted
Dog_house = Not Accepted
 cats = Not Accepted


In [7]:
%pip install automathon

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from automathon import DFA

states = {
    "q0",
    "q1",
    "q_dead"
}

input_symbols = {
    "lowercase",
    "other"
}

transitions = {
    "q0": {
        "lowercase": "q1",
        "other": "q_dead"
    },

    "q1": {
        "lowercase": "q1",
        "other": "q_dead"
    },

    "q_dead": {
        "lowercase": "q_dead",
        "other": "q_dead"
    }
}

initial_state = "q0"

final_states = {
    "q1"
}

dfa_visual = DFA(
    states,
    input_symbols,
    transitions,
    initial_state,
    final_states
)

print("DFA created successfully")
print("Valid DFA:", dfa_visual.is_valid())

DFA created successfully
Valid DFA: True


In [9]:
dfa_visual.view("DFA_Visualization")

In [10]:
print("cat:", dfa_visual.accept("cat"))
print("dog:", dfa_visual.accept("dog"))
print("Dog:", dfa_visual.accept("Dog"))
print("dog1:", dfa_visual.accept("dog1"))

cat: False
dog: False
Dog: False
dog1: False


In [11]:
def show_dfa_steps(word):
    state = "q0"

    print("Input:", word)
    print("Start state:", state)

    for ch in word:
        if "a" <= ch <= "z":
            symbol = "lowercase"
        else:
            symbol = "other"

        next_state = dfa_transition[state][symbol]
        print(ch, ":", state, "->", next_state)

        state = next_state

    print("Result:", "Accepted" if state == "q1" else "Not Accepted")

In [12]:
show_dfa_steps("cat")

Input: cat
Start state: q0
c : q0 -> q1
a : q1 -> q1
t : q1 -> q1
Result: Accepted


In [ ]:
#FST

In [13]:
input_alphabet = {
    "word",
    "root+s",
    "root+es",
    "root-y+ies"
}

print("Input Alphabet:")
print(input_alphabet)

Input Alphabet:
{'word', 'root+s', 'root+es', 'root-y+ies'}


In [14]:
output_alphabet = {
    "root",
    "N",
    "SG",
    "PL"
}

print("Output Alphabet:")
print(output_alphabet)

Output Alphabet:
{'N', 'PL', 'SG', 'root'}


In [15]:
fst_transition = [
    ("q0", "word", "word+N+SG", "qF"),
    ("q0", "root+s", "root+N+PL", "qS"),
    ("q0", "root+es", "root+N+PL", "qE"),
    ("q0", "root-y+ies", "root+N+PL", "qY")
]

print("FST Transition Table")
print("--------------------")
print("Current State | Input | Output | Next State")

for row in fst_transition:
    print(row)

FST Transition Table
--------------------
Current State | Input | Output | Next State
('q0', 'word', 'word+N+SG', 'qF')
('q0', 'root+s', 'root+N+PL', 'qS')
('q0', 'root+es', 'root+N+PL', 'qE')
('q0', 'root-y+ies', 'root+N+PL', 'qY')


In [16]:
plural_rules = {
    "E insertion": ["s", "z", "x", "ch", "sh"],
    "Y replacement": ["y"],
    "S addition": ["default"]
}

print("Morphological Rules")
print("-------------------")

for rule, endings in plural_rules.items():
    print(rule, ":", endings)

Morphological Rules
-------------------
E insertion : ['s', 'z', 'x', 'ch', 'sh']
Y replacement : ['y']
S addition : ['default']


In [17]:
def find_plural_root(word):

    # E insertion
    endings = ["s", "z", "x", "ch", "sh"]

    if word.endswith("es"):

        possible_root = word[:-2]

        for ending in endings:

            if possible_root.endswith(ending):

                if possible_root in noun_lexicon:
                    return possible_root, "E insertion"

    # Y replacement
    if word.endswith("ies"):

        possible_root = word[:-3] + "y"

        if possible_root in noun_lexicon:
            return possible_root, "Y replacement"

    # S addition
    if word.endswith("s") and not word.endswith("ss"):

        possible_root = word[:-1]

        if possible_root in noun_lexicon:
            return possible_root, "S addition"

    return None, None

In [18]:
def fst_analyze(word):

    if dfa_check(word) != "Accepted":
        return "Invalid Word"

    if word in noun_lexicon:

        root, rule = find_plural_root(word)

        if root is None:
            return word + "+N+SG"

    root, rule = find_plural_root(word)

    if root is not None:
        return root + "+N+PL"

    return "Invalid Word"

In [19]:
test_fst = [
    "fox",
    "foxes",
    "watch",
    "watches",
    "try",
    "tries",
    "bag",
    "bags",
    "foxs"
]

for word in test_fst:
    print(word, "=", fst_analyze(word))

fox = fox+N+SG
foxes = fox+N+PL
watch = watch+N+SG
watches = watch+N+PL
try = try+N+SG
tries = try+N+PL
bag = bag+N+SG
bags = bag+N+PL
foxs = fox+N+PL


In [20]:
invalid_words = [
    "foxs",
    "dog1",
    "1dog",
    "Dog",
    "dog_house",
    "cat!",
    "hello world"
]

for word in invalid_words:
    print(word, "=", fst_analyze(word))

foxs = fox+N+PL
dog1 = Invalid Word
1dog = Invalid Word
Dog = Invalid Word
dog_house = Invalid Word
cat! = Invalid Word
hello world = Invalid Word


In [21]:
examples = {
    "E insertion": [
        "foxes",
        "watches",
        "boxes"
    ],

    "Y replacement": [
        "tries",
        "parties"
    ],

    "S addition": [
        "bags",
        "cars",
        "books"
    ]
}

for rule, test_list in examples.items():

    print("\n" + rule)

    for word in test_list:
        print(word, "=", fst_analyze(word))


E insertion
foxes = fox+N+PL
watches = watch+N+PL
boxes = box+N+PL

Y replacement
tries = try+N+PL
parties = party+N+PL

S addition
bags = bag+N+PL
cars = car+N+PL
books = book+N+PL


In [22]:
results = {}

for word in sorted(noun_lexicon):
    results[word] = fst_analyze(word)

print("Total unique nouns:", len(results))

Total unique nouns: 19287


In [23]:
count = 0

for word, output in results.items():

    print(word, "=", output)

    count += 1

    if count == 50:
        break

$.027 = Invalid Word
$.03 = Invalid Word
$.054/mbf = Invalid Word
$.07 = Invalid Word
$.07/cwt = Invalid Word
$.076 = Invalid Word
$.09 = Invalid Word
$.105 = Invalid Word
$.12 = Invalid Word
$.30 = Invalid Word
$.30/mbf = Invalid Word
$.50 = Invalid Word
$.65 = Invalid Word
$.75 = Invalid Word
$.80 = Invalid Word
$.86 = Invalid Word
$.90 = Invalid Word
$0.9 = Invalid Word
$1,000 = Invalid Word
$1,000,000 = Invalid Word
$1,000,000,000 = Invalid Word
$1,200 = Invalid Word
$1,250,000 = Invalid Word
$1,276 = Invalid Word
$1,390 = Invalid Word
$1,450,000,000 = Invalid Word
$1,500 = Invalid Word
$1,500,000 = Invalid Word
$1,600 = Invalid Word
$1,750,000 = Invalid Word
$1,800 = Invalid Word
$1,961,000 = Invalid Word
$1.0 = Invalid Word
$1.00 = Invalid Word
$1.1 = Invalid Word
$1.10 = Invalid Word
$1.26 = Invalid Word
$1.4 = Invalid Word
$1.5 = Invalid Word
$1.6 = Invalid Word
$1.60 = Invalid Word
$1.65 = Invalid Word
$1.7 = Invalid Word
$1.8 = Invalid Word
$1.80 = Invalid Word
$1.9 = Invalid

In [24]:
singular_count = 0
plural_count = 0
invalid_count = 0

for output in results.values():

    if output.endswith("+N+SG"):
        singular_count += 1

    elif output.endswith("+N+PL"):
        plural_count += 1

    else:
        invalid_count += 1

print("Singular:", singular_count)
print("Plural:", plural_count)
print("Invalid:", invalid_count)

Singular: 12710
Plural: 4343
Invalid: 2234


In [25]:
count = 0

for word, output in results.items():

    if output.endswith("+N+PL"):

        print(word, "=", output)

        count += 1

        if count == 30:
            break

abbreviations = abbreviation+N+PL
aberrations = aberration+N+PL
abilities = ability+N+PL
abolitionists = abolitionist+N+PL
aborigines = aborigine+N+PL
abortions = abortion+N+PL
absences = absence+N+PL
absolutes = absolute+N+PL
absorptions = absorption+N+PL
abstractions = abstraction+N+PL
abstracts = abstract+N+PL
absurdities = absurdity+N+PL
abuses = abuse+N+PL
academics = academic+N+PL
academies = academy+N+PL
accelerations = acceleration+N+PL
accelerators = accelerator+N+PL
accelerometers = accelerometer+N+PL
accents = accent+N+PL
accesses = access+N+PL
accessories = accessory+N+PL
accidents = accident+N+PL
accolades = accolade+N+PL
accommodations = accommodation+N+PL
accompaniments = accompaniment+N+PL
accompanists = accompanist+N+PL
accomplices = accomplice+N+PL
accomplishments = accomplishment+N+PL
accountants = accountant+N+PL
accounts = account+N+PL


In [26]:
count = 0

for word, output in results.items():

    if output == "Invalid Word":

        print(word, "=", output)

        count += 1

        if count == 30:
            break

$.027 = Invalid Word
$.03 = Invalid Word
$.054/mbf = Invalid Word
$.07 = Invalid Word
$.07/cwt = Invalid Word
$.076 = Invalid Word
$.09 = Invalid Word
$.105 = Invalid Word
$.12 = Invalid Word
$.30 = Invalid Word
$.30/mbf = Invalid Word
$.50 = Invalid Word
$.65 = Invalid Word
$.75 = Invalid Word
$.80 = Invalid Word
$.86 = Invalid Word
$.90 = Invalid Word
$0.9 = Invalid Word
$1,000 = Invalid Word
$1,000,000 = Invalid Word
$1,000,000,000 = Invalid Word
$1,200 = Invalid Word
$1,250,000 = Invalid Word
$1,276 = Invalid Word
$1,390 = Invalid Word
$1,450,000,000 = Invalid Word
$1,500 = Invalid Word
$1,500,000 = Invalid Word
$1,600 = Invalid Word
$1,750,000 = Invalid Word


In [27]:
final_tests = [
    "cat",
    "cats",
    "fox",
    "foxes",
    "try",
    "tries",
    "bag",
    "bags",
    "foxs",
    "dog1",
    "Dog"
]

print("Final FST Results")
print("-----------------")

for word in final_tests:
    print(word, "=", fst_analyze(word))

Final FST Results
-----------------
cat = cat+N+SG
cats = cat+N+PL
fox = fox+N+SG
foxes = fox+N+PL
try = try+N+SG
tries = try+N+PL
bag = bag+N+SG
bags = bag+N+PL
foxs = fox+N+PL
dog1 = Invalid Word
Dog = Invalid Word
